This script is used to create Asset Catalog of Wires, specifically Overhead Transmission Lines. In the cim model of wires, the main parameters required include radius and count of strands and core strands. The AC and DC resistances per length is another important parameter. The focus of this script will be mainly extracting these parameters. 

Mainly three OEMs for wires were identified as follows.

1. Southwire A - https://www.southwire.com/wire-cable/bare-aluminum-overhead-transmission-distribution/c/c-bare-aluminum-overhead
2. Prysmian UK data sheets - https://na.prysmian.com/markets/power-grid/overhead-transmission
3. Nexans - https://www.nexans.us/en/Product-Datasheets/Utility-Transmission-Distribution/Bare-Overhead-Conductors.html

This doc will focus only on Southwire wires, as it covers most types of wires.

Aluminum overhead wires from Southwire is targetted. The data available from OEM website is extracted into a csv file (tabular format) using python scripts. The csv files are used as inputs to the following python script to create a cim based assset catalog for these wires.

In [76]:
## Importing neccessary libraries
import cimgraph.data_profile.cim17v40 as cim
import numpy as np
import pandas as pd
import os
import cimgraph.data_profile.cim17v40 as cim
from cimgraph.models import FeederModel
from cimgraph.databases import XMLFile
from cimgraph import utils
from mermaid import Mermaid

### Initialize cim profile and an empty XML file
os.environ['CIMG_CIM_PROFILE'] = 'cimgraph.data_profile.cim17v40'
namespaces = {'cim':'http://iec.ch/TC57/CIM100#',
              'gad':'http://gridappsd.org/CIM/extension#'}
file = XMLFile(filename='', namespaces=namespaces)
network = FeederModel(container=None, connection=file)


File  not found. Defaulting to empty network graph
No root element found in XML file


In [77]:
## Importing neccessaruy libraries
import cimgraph.data_profile.cim17v40 as cim
import os
import cimgraph.data_profile.cim17v40 as cim
from cimgraph.models import FeederModel
from cimgraph.databases import XMLFile
from cimgraph import utils
from mermaid import Mermaid
import numpy as np
import pandas as pd
### Initialize cim profile and an empty XML file
os.environ['CIMG_CIM_PROFILE'] = 'cimgraph.data_profile.cim17v40'
namespaces = {'cim':'http://iec.ch/TC57/CIM100#',
              'gad':'http://gridappsd.org/CIM/extension#'}
file = XMLFile(filename='', namespaces=namespaces)
network = FeederModel(container=None, connection=file)

## reading the csv file with wire parameters (extracted from OEM website)
df_SouthWire_para_table = pd.read_csv('ACSR_TL_para_Tables_Cleaned.csv') ## link to OEM website: https://assets.southwire.com/ImConvServlet/imconv/6e40b948ad8bbb2c69490138659678cbf373c912/origin?hybrisId=otmmHybrisPRD&assetDescr=ACSR-Dec-2020
np_SouthWire_para_table = df_SouthWire_para_table.values[ 2:, :]


## exttracting strand/ core strand and splitting them
Stranding_AI_stl = np_SouthWire_para_table[:,2]
split_arr = np.array([s.split('/') for s in Stranding_AI_stl], dtype=int)
Aluminium_strand_count = split_arr[:, 0]   # first column
Steel_strand_count = split_arr[:, 1]  # second column


# for k in range(np_SouthWire_para_table.shape[0]): 
for k in range(np_SouthWire_para_table.shape[0]): 
    ## 1. Extracting wire parametes from csv
    Code_word = np_SouthWire_para_table[k,0]
    Size_Description = np_SouthWire_para_table[k,1]
    Strand_Count = Aluminium_strand_count[k]
    Core_Strand_Count = Steel_strand_count[k]
    Strand_radius = float(np_SouthWire_para_table[k,3])/2 ## in
    Core_Strand_radius = float(np_SouthWire_para_table[k,4])/2 ## in
    Core_radius = float(np_SouthWire_para_table[k,5])/2 ## in
    Radius = float(np_SouthWire_para_table[k,6])/2 ## in
    Mass_per_length = float(np_SouthWire_para_table[k,9])/1000 ## lbs/ ft
    Rated_Strength = float(np_SouthWire_para_table[k,12]) ## lbs
    Resistance_per_length_rDC_20 = float(np_SouthWire_para_table[k,13]) ## ohms/ kilo feet
    Resistance_per_length_rAC_75 = float(np_SouthWire_para_table[k,14]) ## ohms/ kilo feet
    Rated_current = float(np_SouthWire_para_table[k,15]) ## Amps
    
    
    ## 2. Assigning the extracted values to the CIM variables
    wire = cim.OverheadWireInfo(name=str(Code_word))
    wire.sizeDescription = Size_Description
    # wire.strandCount = Strand_Count
    # wire.coreStrandCount = Core_Strand_Count
    # wire.strandRadius = cim.Length(value = Strand_radius, input_unit = 'in')
    # wire.coreStrandRadius = cim.Length(value = Core_Strand_radius, input_unit = 'in')
    wire.coreRadius = cim.Length(value = Core_radius, input_unit = 'in')
    wire.radius = cim.Length(value= Radius, input_unit='in')
    # wire.massPerLength = cim.MassPerLength(value= Mass_per_length, input_unit='lb/ft')
    # wire.ratedStrength = cim.Force(value= Rated_Strength, input_unit='lb')
    wire.rDC20 = cim.ResistancePerLength(value= Resistance_per_length_rDC_20, input_unit='ohm/kft')
    wire.rAC75 = cim.ResistancePerLength(value= Resistance_per_length_rAC_75, input_unit='ohm/kft')
    wire.ratedCurrent = cim.CurrentFlow(value= Rated_current, input_unit='amp')
    # wire.pprint()

    network.add_to_graph( wire)


#network.add_to_graph( wire)



# print(network)
# ### Saving the wires (over head transmission line) asset catalog into xml file
# utils.write_xml(network, 'Wires_Asset_Catalog_1.xml', namespaces)

File  not found. Defaulting to empty network graph
No root element found in XML file


In [78]:
## reading the csv file with wire parameters (extracted from OEM website)
## AAC - All Aluminum Conductor. Bare.
df_SouthWire_para_table = pd.read_csv('AAC_Southwire_to_ACSR_schema_WITH_STRUCTURE.csv') ## link to OEM website: https://assets.southwire.com/ImConvServlet/imconv/6e40b948ad8bbb2c69490138659678cbf373c912/origin?hybrisId=otmmHybrisPRD&assetDescr=ACSR-Dec-2020
np_SouthWire_para_table = df_SouthWire_para_table.values[ 2:, :]



# for k in range(np_SouthWire_para_table.shape[0]): 
for k in range(np_SouthWire_para_table.shape[0]): 
    ## 1. Extracting wire parametes from csv
    Code_word = np_SouthWire_para_table[k,0]
    Radius = float(np_SouthWire_para_table[k,6])/2 ## in
    Resistance_per_length_rDC_20 = float(np_SouthWire_para_table[k,13]) ## ohms/ kilo feet
    Resistance_per_length_rAC_75 = float(np_SouthWire_para_table[k,14]) ## ohms/ kilo feet
    Rated_current = float(np_SouthWire_para_table[k,15]) ## Amps
    
    ## 2. Assigning the extracted values to the CIM variables
    wire = cim.OverheadWireInfo(name=str(Code_word))
    wire.radius = cim.Length(value= Radius, input_unit='in')
    wire.rDC20 = cim.ResistancePerLength(value= Resistance_per_length_rDC_20, input_unit='ohm/kft')
    wire.rAC75 = cim.ResistancePerLength(value= Resistance_per_length_rAC_75, input_unit='ohm/kft')
    wire.ratedCurrent = cim.CurrentFlow(value= Rated_current, input_unit='amp')

    network.add_to_graph( wire)


# ### Saving the wires (over head transmission line) asset catalog into xml file
# utils.write_xml(network, 'Wires_Asset_Catalog_1.xml', namespaces)

In [79]:

## reading the csv file with wire parameters (extracted from OEM website)
## ACSS - Aluminum Conductor, Steel Supported. Bare.
df_SouthWire_para_table = pd.read_csv('ACSS_to_ACSR_schema_WITH_STRUCTURE.csv') ## link to OEM website: https://assets.southwire.com/ImConvServlet/imconv/dd014b54f1bf8e2670fc98f2c2170d39b2bd5b1c/origin?hybrisId=otmmHybrisPRD&assetDescr=ACSS%20-%2005-22-2015
np_SouthWire_para_table = df_SouthWire_para_table.values[ 2:, :]



# for k in range(np_SouthWire_para_table.shape[0]): 
for k in range(np_SouthWire_para_table.shape[0]): # 
    ## 1. Extracting wire parametes from csv
    Code_word = np_SouthWire_para_table[k,0]
    Size_Description = np_SouthWire_para_table[k,1]
    
 
    Core_radius = float(np_SouthWire_para_table[k,5])/2 ## in
    Radius = float(np_SouthWire_para_table[k,6])/2 ## in
   
    Resistance_per_length_rDC_20 = float(np_SouthWire_para_table[k,13]) ## ohms/ kilo feet
    Resistance_per_length_rAC_75 = float(np_SouthWire_para_table[k,14]) ## ohms/ kilo feet
    Rated_current = float(np_SouthWire_para_table[k,15]) ## Amps
    
    
    ## 2. Assigning the extracted values to the CIM variables
    wire = cim.OverheadWireInfo(name=str(Code_word))
    wire.sizeDescription = Size_Description

    #wire.coreRadius = cim.Length(value = Core_radius, input_unit = 'in')
    wire.radius = cim.Length(value= Radius, input_unit='in')

    wire.rDC20 = cim.ResistancePerLength(value= Resistance_per_length_rDC_20, input_unit='ohm/kft')
    wire.rAC75 = cim.ResistancePerLength(value= Resistance_per_length_rAC_75, input_unit='ohm/kft')
    wire.ratedCurrent = cim.CurrentFlow(value= Rated_current, input_unit='amp')
    # wire.pprint()

    network.add_to_graph( wire)

# ### Saving the wires (over head transmission line) asset catalog into xml file
# utils.write_xml(network, 'Wires_Asset_Catalog_1.xml', namespaces)


In [80]:
## reading the csv file with wire parameters (extracted from OEM website)
## ACSS/TW - Aluminum Conductor, Steel Supported. Trapezoidal Shaped Aluminum Strands. Bare.
df_SouthWire_para_table = pd.read_csv('ACSS_TW_to_ACSR_schema_WITH_STRUCTURE.csv') ## link to OEM website: https://assets.southwire.com/ImConvServlet/imconv/3339cca43748ad8dff5b296232a256aa970621c0/origin?hybrisId=otmmHybrisPRD&assetDescr=ACSS%20TW
np_SouthWire_para_table = df_SouthWire_para_table.values[ 2:, :]


# for k in range(np_SouthWire_para_table.shape[0]): 
for k in range(np_SouthWire_para_table.shape[0]): # 
    ## 1. Extracting wire parametes from csv
    Code_word = np_SouthWire_para_table[k,0]
    Size_Description = np_SouthWire_para_table[k,1]
    
 
    Core_radius = float(np_SouthWire_para_table[k,5])/2 ## in
    Radius = float(np_SouthWire_para_table[k,6])/2 ## in
   
    Resistance_per_length_rDC_20 = float(np_SouthWire_para_table[k,13]) ## ohms/ kilo feet
    Resistance_per_length_rAC_75 = float(np_SouthWire_para_table[k,14]) ## ohms/ kilo feet
    Rated_current = float(np_SouthWire_para_table[k,15]) ## Amps
    
    
    ## 2. Assigning the extracted values to the CIM variables
    wire = cim.OverheadWireInfo(name=str(Code_word))
    wire.sizeDescription = Size_Description

    #wire.coreRadius = cim.Length(value = Core_radius, input_unit = 'in')
    wire.radius = cim.Length(value= Radius, input_unit='in')

    wire.rDC20 = cim.ResistancePerLength(value= Resistance_per_length_rDC_20, input_unit='ohm/kft')
    wire.rAC75 = cim.ResistancePerLength(value= Resistance_per_length_rAC_75, input_unit='ohm/kft')
    wire.ratedCurrent = cim.CurrentFlow(value= Rated_current, input_unit='amp')
    # wire.pprint()

    network.add_to_graph( wire)

# ### Saving the wires (over head transmission line) asset catalog into xml file
# utils.write_xml(network, 'Wires_Asset_Catalog_1.xml', namespaces)




In [81]:
## reading the csv file with wire parameters (extracted from OEM website)
## ACSR/AW - Aluminum Conductor. Steel Reinforced . Bare
df_SouthWire_para_table = pd.read_csv('ACSR_AW_to_ACSR_schema_WITH_STRUCTURE.csv') ## link to OEM website: https://assets.southwire.com/ImConvServlet/imconv/75e095031f95f62bb00f07c24da2a731832b3213/origin?hybrisId=otmmHybrisPRD&assetDescr=ACSR-AW
np_SouthWire_para_table = df_SouthWire_para_table.values[ 2:, :]

# for k in range(np_SouthWire_para_table.shape[0]): 
for k in range(np_SouthWire_para_table.shape[0]): # 
    ## 1. Extracting wire parametes from csv
    Code_word = np_SouthWire_para_table[k,0]
    Size_Description = np_SouthWire_para_table[k,1]
    
 
    Core_radius = float(np_SouthWire_para_table[k,5])/2 ## in
    Radius = float(np_SouthWire_para_table[k,6])/2 ## in
   
    Resistance_per_length_rDC_20 = float(np_SouthWire_para_table[k,13]) ## ohms/ kilo feet
    Resistance_per_length_rAC_75 = float(np_SouthWire_para_table[k,14]) ## ohms/ kilo feet
    Rated_current = float(np_SouthWire_para_table[k,15]) ## Amps
    
    
    ## 2. Assigning the extracted values to the CIM variables
    wire = cim.OverheadWireInfo(name=str(Code_word))
    wire.sizeDescription = Size_Description

    #wire.coreRadius = cim.Length(value = Core_radius, input_unit = 'in')
    wire.radius = cim.Length(value= Radius, input_unit='in')

    wire.rDC20 = cim.ResistancePerLength(value= Resistance_per_length_rDC_20, input_unit='ohm/kft')
    wire.rAC75 = cim.ResistancePerLength(value= Resistance_per_length_rAC_75, input_unit='ohm/kft')
    wire.ratedCurrent = cim.CurrentFlow(value= Rated_current, input_unit='amp')
    # wire.pprint()

    network.add_to_graph( wire)

# ### Saving the wires (over head transmission line) asset catalog into xml file
# utils.write_xml(network, 'Wires_Asset_Catalog_1.xml', namespaces)


In [82]:
## reading the csv file with wire parameters (extracted from OEM website)
## ACSR/TW - Aluminum Conductor. Steel Reinforced. Trapezoidal Shaped Aluminum Strands.
df_SouthWire_para_table = pd.read_csv('ACSR_TW_to_ACSR_schema_WITH_STRUCTURE.csv') ## link to OEM website: https://assets.southwire.com/ImConvServlet/imconv/c1b18d4932e34a12f603ceb16dc472bc09dd35ff/origin?hybrisId=otmmHybrisPRD&assetDescr=ACSR_TW
np_SouthWire_para_table = df_SouthWire_para_table.values[ 2:, :]


# for k in range(np_SouthWire_para_table.shape[0]): 
for k in range(np_SouthWire_para_table.shape[0]): # 
    ## 1. Extracting wire parametes from csv
    Code_word = np_SouthWire_para_table[k,0]
    Size_Description = np_SouthWire_para_table[k,1]
    
 
    Core_radius = float(np_SouthWire_para_table[k,5])/2 ## in
    Radius = float(np_SouthWire_para_table[k,6])/2 ## in
   
    Resistance_per_length_rDC_20 = float(np_SouthWire_para_table[k,13]) ## ohms/ kilo feet
    Resistance_per_length_rAC_75 = float(np_SouthWire_para_table[k,14]) ## ohms/ kilo feet
    Rated_current = float(np_SouthWire_para_table[k,15]) ## Amps
    
    
    ## 2. Assigning the extracted values to the CIM variables
    wire = cim.OverheadWireInfo(name=str(Code_word))
    wire.sizeDescription = Size_Description

    #wire.coreRadius = cim.Length(value = Core_radius, input_unit = 'in')
    wire.radius = cim.Length(value= Radius, input_unit='in')

    wire.rDC20 = cim.ResistancePerLength(value= Resistance_per_length_rDC_20, input_unit='ohm/kft')
    wire.rAC75 = cim.ResistancePerLength(value= Resistance_per_length_rAC_75, input_unit='ohm/kft')
    wire.ratedCurrent = cim.CurrentFlow(value= Rated_current, input_unit='amp')
    # wire.pprint()

    network.add_to_graph( wire)

# ### Saving the wires (over head transmission line) asset catalog into xml file
# utils.write_xml(network, 'Wires_Asset_Catalog_1.xml', namespaces)

In [83]:
### Saving the wires (over head transmission line) asset catalog into xml file
utils.write_xml(network, 'Wires_Asset_Catalog_1.xml', namespaces)